# 02 — Weather & Database
Building a DuckDB database and pulling weather data for every Padres home game from the Open-Meteo API (free, no API key needed).

In [18]:
import pandas as pd
import requests

# Load the training data we already collected
df = pd.read_csv("../data/padres_games_raw.csv")
print(f"Loaded {len(df)} games")
df_2026 = pd.read_csv("../data/padres_2026_prediction.csv")
print(f"Loaded {len(df_2026)} 2026 games")
df.head()

Loaded 320 games
Loaded 81 2026 games


,date,season,opponent,home_score,away_score,attendance,status,game_pk
0,2022-04-14,2022,Atlanta Braves,12,1,44844,Final,662237
1,2022-04-15,2022,Atlanta Braves,2,5,41993,Final,662236
2,2022-04-16,2022,Atlanta Braves,2,5,36924,Final,662235
3,2022-04-17,2022,Atlanta Braves,2,1,37694,Final,662234
4,2022-04-18,2022,Cincinnati Reds,4,1,31121,Final,662233


## Pulling Weather for 2022-2025 Game Dates
Petco Park coordinates are latitude 32.7076, longitude -117.1570.

In [15]:
def get_weather(date, lat=32.7076, lon=-117.1570):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": date,
        "end_date": date,
        "hourly": "temperature_2m,precipitation",
        "temperature_unit": "fahrenheit",
        "timezone": "America/Los_Angeles"
    }
    resp = requests.get(url, params=params).json()
    hourly = resp.get("hourly", {})

    # Get conditions around game time (4pm-8pm = hours 16-20)
    temps = [t for t in hourly.get("temperature_2m", [])[16:21] if t is not None]
    precip = [p for p in hourly.get("precipitation", [])[16:21] if p is not None]

    return {
        "date": date,
        "avg_temp_f": round(sum(temps) / len(temps), 1) if temps else None,
        "avg_precip_mm": round(sum(precip) / len(precip), 1) if precip else None
    }

# Pull weather for all game dates
dates = df["date"].unique()
print(f"Pulling weather for {len(dates)} unique game dates...")

weather_data = [get_weather(d) for d in dates]
df_weather = pd.DataFrame(weather_data)
print("Done!")
df_weather.head()


Pulling weather for 317 unique game dates...


KeyboardInterrupt: 

In [5]:
df_weather.to_csv("../data/padres_weather.csv", index=False)
print(f"Saved weather for {len(df_weather)} dates")
print(df_weather.describe())

Saved weather for 317 dates
       avg_temp_f  avg_precip_mm
count  317.000000     317.000000
mean    67.980126       0.014511
std      5.619852       0.140673
min     53.900000       0.000000
25%     63.500000       0.000000
50%     68.700000       0.000000
75%     72.000000       0.000000
max     86.900000       2.100000


In [8]:
# Pull weather for 2026 games already played
dates_2026 = df_2026[df_2026['status'] == 'Final']['date'].unique()
print(f"Pulling weather for {len(dates_2026)} 2026 game dates...")

weather_2026 = [get_weather(d) for d in dates_2026]
df_weather_2026 = pd.DataFrame(weather_2026)
df_weather_2026.to_csv("../data/padres_weather_2026.csv", index=False)
print("Done!")
df_weather_2026.describe()

Pulling weather for 46 2026 game dates...
Done!


,avg_temp_f,avg_precip_mm
count,46.000000,46.000000
mean,66.493478,0.013043
std,2.205891,0.061855
min,62.600000,0.000000
25%,65.100000,0.000000
50%,65.900000,0.000000
75%,67.875000,0.000000
max,72.900000,0.400000


## Build DuckDB Database
Load games and weather data into a DuckDB database. DuckDB uses standard SQL syntax and is optimized for analytical queries — similar to how BigQuery works in production.

In [17]:
import duckdb

# Create database file
con = duckdb.connect("../data/padres.duckdb")

# Load games table
con.execute("CREATE OR REPLACE TABLE games AS SELECT * FROM read_csv_auto('../data/padres_games_raw.csv')")

# Load weather table
con.execute("CREATE OR REPLACE TABLE weather AS SELECT * FROM read_csv_auto('../data/padres_weather.csv')")

print("Tables created:")
print(con.execute("SHOW TABLES").fetchdf())
print("Games rows:", con.execute("SELECT COUNT(*) FROM games").fetchone()[0])
print("Weather rows:", con.execute("SELECT COUNT(*) FROM weather").fetchone()[0])


Tables created:
      name
0    games
1  weather
Games rows: 320
Weather rows: 317


In [19]:
# Join games and weather on date
df_combined = con.execute("""
    SELECT 
        g.date,
        g.season,
        g.opponent,
        g.home_score,
        g.away_score,
        g.attendance,
        g.status,
        g.game_pk,
        w.avg_temp_f,
        w.avg_precip_mm
    FROM games g
    LEFT JOIN weather w ON g.date = w.date
    WHERE g.status = 'Final'
    ORDER BY g.date
""").fetchdf()

print(f"Combined dataset: {len(df_combined)} games")
print(df_combined.head())

Combined dataset: 320 games
        date  season         opponent  home_score  away_score  attendance  \
0 2022-04-14    2022   Atlanta Braves          12           1       44844   
1 2022-04-15    2022   Atlanta Braves           2           5       41993   
2 2022-04-16    2022   Atlanta Braves           2           5       36924   
3 2022-04-17    2022   Atlanta Braves           2           1       37694   
4 2022-04-18    2022  Cincinnati Reds           4           1       31121   

  status  game_pk  avg_temp_f  avg_precip_mm  
0  Final   662237        61.3            0.0  
1  Final   662236        62.6            0.0  
2  Final   662235        62.8            0.0  
3  Final   662234        62.5            0.0  
4  Final   662233        63.6            0.0  


In [20]:
df_combined.to_csv("../data/padres_master.csv", index=False)
print("Saved padres_master.csv")

Saved padres_master.csv
